In [ ]:
from itertools import product
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from scaling.visualize import visualize_train_curves, plot_line_fit, plot_isoflops
from pathlib import Path
from scaling.utils import (
    get_final_points_from_curve_set,
    fit_parametric_form,
    functional_form_chin3,
    fit_parametric_form_parallel
)
from sklearn.metrics import r2_score
from IPython.display import display, Math
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns
sns.set_style("white")

In [ ]:
unique_col_list = ["base_N", "target_N", "tkpm", "shrink", "method"]
y_col = "Validation Loss"
x_col = "flops"

N_TEST = 1217722368
mup_test =  1.5173616409301758
warm_test = 1.496509075164795


def preprocess_warmstarting(df, y_col_to_smooth=None, smoothing_window=100):
    __df = pd.DataFrame()
    for i, x in enumerate(df.groupby(unique_col_list)):
        _df = x[1].sort_values(by="flops")
        # smooth it
        if y_col_to_smooth is not None:
            # +"_smoothed"
            _df[y_col_to_smooth] = _df[y_col_to_smooth].rolling(smoothing_window, win_type='gaussian', min_periods=1).mean(std=smoothing_window / 10)
        
        # scaling tokens and flops to the max
        max_intended_tokens = (_df.iloc[-1]["target_N"] * _df.iloc[-1]["tkpm"])
        if abs((max_intended_tokens -  _df["tokens"].max()) / _df["tokens"].max()) > 0.01:
            print("Wrong tkpm: ", x[0])
            continue
        _df["tokens"] = np.round(max_intended_tokens / _df["tokens"].max() * _df["tokens"])
        
        max_intended_flops = 6. * max_intended_tokens * _df["target_N"]
        _df["flops"] = np.round(max_intended_flops / _df["flops"].max() * _df["flops"])
        
        __df = pd.concat([__df, _df])
    
    # print('Dropping tkpm <= 5')
    # __df = __df[__df['tkpm'] > 5.]
    
    return __df

def preprocess_approach3_data(df, log_transform_y=True):
    N, D, G = df["target_N"].values, df["tokens"].values, df["g"].values
    y = df["Validation Loss"].values

    _df = pd.DataFrame.from_dict({
        "N": N,
        "D": D,
        "G": G,
        "Loss": y
    }).groupby(by=["N", "D", "G"]).min().reset_index()
    _df.sort_values(by=["N", "D", "G"], inplace=True)

    data_X = _df[["N", "D", "G"]].values
    data_y = _df["Loss"].values
    if log_transform_y:
        data_y = np.log(data_y)

    return data_X, data_y

def filter_pairs(df, jump_size=1):
    distinct_values = sorted(
        pd.unique(df[['base_N', 'target_N']].values.ravel())
    )
    pairs = list(zip(distinct_values[:-jump_size], distinct_values[jump_size:]))
    df_filtered = df[df[['base_N', 'target_N']].apply(tuple, axis=1).isin(pairs)]
    return df_filtered

def format_approach3_results(params):
    _a, alpha, _b, beta, _e = params

    A = np.exp(_a)
    B = np.exp(_b)
    E = np.exp(_e)
    s = f"{A:0.2f} N^({-alpha:0.3f}) + {B:0.2f} D^({-beta:0.3f}) + {E:0.3f}"
    print(s)
    return f"{int(A)}", f"{alpha:0.3f}", f"{int(B)}", f"{beta:0.3f}", f"{E:0.3f}"

In [ ]:
warmstarting_df = pd.read_parquet(
    "../data/experimental_warmstarting.parquet", # "../data/warmstart_runs_flattened.parquet"
)
warmstarting_df = warmstarting_df.dropna(subset=[y_col])
warmstarting_df = preprocess_warmstarting(warmstarting_df)

In [ ]:
final_points_df = get_final_points_from_curve_set(
    warmstarting_df,
    unique_col_list,
    x_col="flops",
    y_col="Validation Loss",
    get_pareto=False,
)

final_points_df['g'] = final_points_df["target_N"] / final_points_df["base_N"]

In [ ]:
final_points_df.groupby(by=["tkpm", "method", "target_N"]).count().reset_index()[["tkpm", "method", "target_N"]]

In [ ]:
def fit_chin3(data_X, data_y, dense_grid=False):
    if dense_grid:
        initialization = list(product(
            np.arange(0, 25, 1),
            np.arange(0, 2, 0.1),
            np.arange(0, 25, 1),
            np.arange(0, 2, 0.1),
            np.arange(-1, 1, 0.1)
        ))
    else:
        initialization = list(product(
            np.arange(0, 5, 1),
            np.arange(0, 2, 0.5),
            np.arange(0, 5, 1),
            np.arange(0, 2, 0.5),
            np.arange(-1, 1, 0.5)
        ))

    best_params, best_loss, df = fit_parametric_form(
        functional_form_chin3,
        data_X[:,:2],
        data_y,
        initialization,
    )

    y_true = data_y
    y_pred = functional_form_chin3(data_X[:,:2], best_params)
    r2_log = r2_score(y_true, y_pred)
    r2 = r2_score(np.exp(y_true), np.exp(y_pred))



    test_point = np.array([[N_TEST, 10 * N_TEST]])
    test_prediction = float(np.exp(functional_form_chin3(test_point, best_params))[0])

    print("Fit")
    format_approach3_results(best_params)
    print("Top fits")
    print(df.head())
    print("Quality of Fit")
    print("Test prediction: ", test_prediction, " | muP true value: ", mup_test, " | warm true value: ", warm_test)
    print("Log scaled R2: ", r2_log)
    print("R2: ", r2)

    return best_params

def get_loss_diff(w_X, w_y, m_X, m_y, w_param=None, m_param=None):
    w_df = pd.DataFrame(w_X[:,:2], columns=["N", "D"])
    w_df['L'] = np.exp(w_y)

    m_df = pd.DataFrame(m_X[:,:2], columns=["N", "D"])
    m_df['L'] = np.exp(m_y)

    # join w_df and m_df on N and D, suffixing L with _w and _m
    joined_df = pd.merge(w_df, m_df, on=["N", "D"], suffixes=("_w", "_m"))
    joined_df['L_diff'] = joined_df['L_w'] - joined_df['L_m']

    if w_param is not None and m_param is not None:
        functional_form_chin3(w_X[:,:2], w_param)
        functional_form_chin3(w_X[:,:2], m_param)

    missing = joined_df[joined_df.isna().any(axis=1)]
    if not missing.empty:
        print("Removing NaNs:")
        print(joined_df[joined_df.isna().any(axis=1)])
        joined_df.dropna(subset=['L_diff'], inplace=True)
    print(joined_df)
    return joined_df

In [ ]:
mup_df = final_points_df[final_points_df['method']=='mup']
mup_jump_df = filter_pairs(mup_df, jump_size=1)
mup_data_X, mup_data_y = preprocess_approach3_data(mup_jump_df, log_transform_y=True)

mup_data_X.to_csv("./output/mup_data_X.csv")
mup_data_y.to_csv("./output/mup_data_y.csv")

y_pred = functional_form_chin3(mup_data_X[:,:2], mup_params)
mup_r2 = r2_score(np.exp(mup_data_y), np.exp(y_pred))

In [ ]:
warm_df = final_points_df[(final_points_df['method']=='paws') & (final_points_df['shrinking']==0.4)]
warm_jump_df = filter_pairs(warm_df, jump_size=1)
warm_data_X, warm_data_y = preprocess_approach3_data(warm_jump_df, log_transform_y=True)

warm_data_X.to_csv("./output/warm_data_X.csv")
warm_data_y.to_csv("./output/warm_data_y.csv")

warm_params = fit_chin3(warm_data_X, warm_data_y)

y_pred = functional_form_chin3(warm_data_X[:,:2], warm_params)
warm_r2 = r2_score(np.exp(warm_data_y), np.exp(y_pred))

In [ ]:
sns.set_context("notebook", font_scale=1.475)
fig, ax = plt.subplots(1, 1, figsize=(6, 4.75), layout='constrained')

D_contour = np.linspace(final_points_df["tokens"].min()*0.85, 2e12, 3000)
N_contour = np.linspace(final_points_df["target_N"].min()*0.85, 1e10, 3000)

N_cg , D_cg = np.meshgrid(N_contour, D_contour)
L_cg = np.zeros_like(N_cg)



for params, factor in [(mup_params, -1), (warm_params, 1)]:
    _L_cg = np.exp(functional_form_chin3(
            np.column_stack((
                N_cg.reshape(-1),
                D_cg.reshape(-1)
            )),
            params
        ))
    L_cg += _L_cg.reshape(N_cg.shape) * factor

norm = TwoSlopeNorm(vmin=L_cg.min(), vcenter=0, vmax=L_cg.max())
cs = ax.contourf(N_cg, D_cg, L_cg, levels=300, cmap="RdBu_r", norm=norm, zorder=-1)
ax.plot(N_contour, N_contour*20, color='black', linewidth=2, zorder=-1, alpha=0.3)
lime_N_D = ax.contour(N_cg, D_cg, L_cg, levels=[0], colors='lime', linewidths=4, zorder=1, alpha=0.5)
# ax.contour(N_cg, D_cg, L_cg, levels=[0], colors='green', linewidths=1.5, zorder=1)

# ax.scatter(warm_data_X[:,0], warm_data_X[:,1], color="black")
L_diff_df = get_loss_diff(warm_data_X, warm_data_y, mup_data_X, mup_data_y)
ax.scatter(L_diff_df["N"], L_diff_df["D"], c=L_diff_df["L_diff"], cmap="RdBu_r", norm=norm, edgecolor="k", linewidth=0.5,)



fig.colorbar(cs, ax=ax, label=r'Loss Difference ($\text{warmstart} - \mu P$)')
# ax.set_title("SZP")
ax.set_xlabel("Parameters")
ax.set_ylabel("Tokens")
ax.loglog()
fig.suptitle("SZP")
fig.savefig(f"figures/neurips_chinchilla_paws.pdf")

In [ ]:
n2n_df = final_points_df[(final_points_df['method']=='net2net')]
n2n_jump_df = filter_pairs(n2n_df, jump_size=1)
n2n_data_X, n2n_data_y = preprocess_approach3_data(n2n_jump_df, log_transform_y=True)
# save numpy data
n2n_data_X.to_csv("./output/n2n_data_X.csv")
n2n_data_y.to_csv("./output/n2n_data_y.csv")

n2n_params = fit_chin3(n2n_data_X, n2n_data_y)

y_pred = functional_form_chin3(n2n_data_X[:,:2], n2n_params)
n2n_r2 = r2_score(np.exp(n2n_data_y), np.exp(y_pred))

In [ ]:
sns.set_context("notebook", font_scale=1.475)

fig, ax = plt.subplots(1, 1, figsize=(6, 4.75), layout='constrained')

D_contour = np.linspace(final_points_df["tokens"].min()*0.85, 2e12, 3000)
N_contour = np.linspace(final_points_df["target_N"].min()*0.85, 1e10, 3000)

N_cg , D_cg = np.meshgrid(N_contour, D_contour)
L_cg = np.zeros_like(N_cg)



for params, factor in [(mup_params, -1), (n2n_params, 1)]:
    _L_cg = np.exp(functional_form_chin3(
            np.column_stack((
                N_cg.reshape(-1),
                D_cg.reshape(-1)
            )),
            params
        ))
    L_cg += _L_cg.reshape(N_cg.shape) * factor

norm = TwoSlopeNorm(vmin=min(L_cg.min(), -0.001), vcenter=0, vmax=max(L_cg.max(),0.001))
cs = ax.contourf(N_cg, D_cg, L_cg, levels=30, cmap="RdBu_r", norm=norm, zorder=-1)
ax.plot(N_contour, N_contour*20, color='black', linewidth=2, zorder=-1, alpha=0.3)
lime_N_D = ax.contour(N_cg, D_cg, L_cg, levels=[0], colors='lime', linewidths=4, zorder=1, alpha=0.5)
# ax.contour(N_cg, D_cg, L_cg, levels=[0], colors='green', linewidths=1.5, zorder=1)

# ax.scatter(n2n_data_X[:,0], n2n_data_X[:,1], color="black")
L_diff_df = get_loss_diff(n2n_data_X, n2n_data_y, mup_data_X, mup_data_y)
ax.scatter(L_diff_df["N"], L_diff_df["D"], c=L_diff_df["L_diff"], cmap="RdBu_r", norm=norm, edgecolor="k", linewidth=0.5,)

fig.colorbar(cs, ax=ax, label=r'Loss Difference ($\text{warmstart} - \mu P$)')
ax.set_xlabel("Parameters")
ax.set_ylabel("Tokens")
ax.loglog()
fig.suptitle("Net2Net")
fig.savefig(f"figures/neurips_chinchilla_net2net.pdf")

In [ ]:
grid_steps = 250

D_contour = np.linspace(warm_data_X[:,1].min()*0.85, warm_data_X[:,1].max()*10, grid_steps)
N_contour = np.linspace(warm_data_X[:,0].min()*0.85, warm_data_X[:,0].max()*10, grid_steps)

N_cg_warm , D_cg_warm = np.meshgrid(N_contour, D_contour)
L_cg_warm = np.zeros_like(N_cg_warm)

for params, factor in [(mup_params, -1), (warm_params, 1)]:
    _L_cg_snp_mlp = np.exp(functional_form_chin3(
            np.column_stack((
                N_cg_warm.reshape(-1),
                D_cg_warm.reshape(-1)
            )),
            params
        ))
    L_cg_warm += _L_cg_snp_mlp.reshape(N_cg_warm.shape) * factor


D_contour = np.linspace(n2n_data_X[:,1].min()*0.85, n2n_data_X[:,1].max()*10, grid_steps)
N_contour = np.linspace(n2n_data_X[:,0].min()*0.85, n2n_data_X[:,0].max()*10, grid_steps)

N_cg_n2n , D_cg_n2n = np.meshgrid(N_contour, D_contour)
L_cg_n2n = np.zeros_like(N_cg_n2n)

for params, factor in [(mup_params, -1), (n2n_params, 1)]:
    _L_cg_n2n = np.exp(functional_form_chin3(
            np.column_stack((
                N_cg_n2n.reshape(-1),
                D_cg_n2n.reshape(-1)
            )),
            params
        ))
    L_cg_n2n += _L_cg_n2n.reshape(L_cg_n2n.shape) * factor

In [ ]:
import seaborn as sns
sns.set_style("white")
sns.set_context("notebook", font_scale=1.475)
fig, axes = plt.subplots(1, 2, figsize=(5*2, 4.75), layout='constrained')

vmin = float(min(L_cg_warm.min(), L_cg_n2n.min()))
vmax = float(max(L_cg_warm.max(), L_cg_n2n.max()))
norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
levels = np.linspace(vmin, vmax, 50)
cmap = "RdBu_r"

# SZP
ax = axes[0]
cs = ax.contourf(N_cg_warm, D_cg_warm, L_cg_warm, levels=levels, cmap=cmap, norm=norm, zorder=-1)
ax.plot(N_contour, N_contour*20, color='black', linewidth=2, zorder=-1, alpha=0.3)
lime_N_D = ax.contour(N_cg_warm, D_cg_warm, L_cg_warm, levels=[0], colors='lime', linewidths=4, zorder=1, alpha=0.5)
# ax.contour(N_cg, D_cg, L_cg, levels=[0], colors='green', linewidths=1.5, zorder=1)

# ax.scatter(snp_mlp_data_X[:,0], snp_mlp_data_X[:,1], color="black")
L_diff_df = get_loss_diff(warm_data_X, warm_data_y, mup_data_X, mup_data_y)
ax.scatter(L_diff_df["N"], L_diff_df["D"], c=L_diff_df["L_diff"], cmap=cmap, norm=norm, edgecolor="k", linewidth=0.5,)
ax.loglog()
ax.set_title("SZP")

# N2N
ax = axes[1]
cs = ax.contourf(N_cg_n2n, D_cg_n2n, L_cg_n2n, levels=levels, cmap=cmap, norm=norm, zorder=-1)
ax.plot(N_contour, N_contour*20, color='black', linewidth=2, zorder=-1, alpha=0.3)
lime_N_D = ax.contour(N_cg_n2n, D_cg_n2n, L_cg_n2n, levels=[0], colors='lime', linewidths=4, zorder=1, alpha=0.5)
# ax.contour(N_cg, D_cg, L_cg, levels=[0], colors='green', linewidths=1.5, zorder=1)

# ax.scatter(n2n_mlp_data_X[:,0], n2n_mlp_data_X[:,1], color="black")
L_diff_df = get_loss_diff(n2n_data_X, n2n_data_y, mup_data_X, mup_data_y)
ax.scatter(L_diff_df["N"], L_diff_df["D"], c=L_diff_df["L_diff"], cmap=cmap, norm=norm, edgecolor="k", linewidth=0.5,)
ax.loglog()
ax.set_title("Net2Net")


fig.colorbar(cs, ax=ax, label=r'Loss Difference ($\text{warmstart} - \mu P$)')
fig.supxlabel("Parameters")
fig.supylabel("Tokens")
fig.suptitle("LLM")
fig.savefig(f"figures/neurips_chinchilla_llm_low_resolution.pdf")

# MLP Setting

In [ ]:
mlp_df = pd.read_parquet(
    "../data/mlp_results.parquet",
)
width_scale_to_params = {
    0: 0.1e6,
    1: 0.1e6,
    2: 0.2e6,
    3: 0.4e6,
    4: 0.9e6,
    5: 1.8e6,
    6: 3.5e6,
    7: 7.0e6,
    8: 14.2e6,
    9: 25.2e6,
    10: 100.7e6,
}
width_scale_to_hidden_dim = {
    0: 48,
    1: 68,
    2: 96,
    3: 136,
    4: 192,
    5: 272,
    6: 384,
    7: 540,
    8: 768,
    9: 1024,
    10: 2048,
}

# create base_N and target_N columns
mlp_df['base_N'] = mlp_df['base_scale'].map(lambda x: width_scale_to_hidden_dim.get(x, np.nan))
mlp_df['target_N'] = mlp_df['target_scale'].map(lambda x: width_scale_to_hidden_dim.get(x, np.nan))
mlp_df['max_flops'] = mlp_df['curve_flops'].map(lambda x: x[-1])

def get_loss_at_flops_mlp(df: pd.DataFrame, flop_intervals: list[float]) -> pd.Series:
    """Get the loss at a specific flop value by interpolation."""
    best_learning_curve = None
    best_final_loss = float('inf')

    # iterate over rows
    for row in df.itertuples(index=False):
        final_loss = row.curve_val[-1]
        if final_loss < best_final_loss:
            best_final_loss = final_loss
            best_learning_curve = pd.Series(
                data=row.curve_val,
                index=row.curve_flops
            )

    # add the flops into the Series if not present
    for flop in flop_intervals:
        if flop not in best_learning_curve.index:
            best_learning_curve.loc[flop] = np.nan
    best_learning_curve = best_learning_curve.sort_index()
    # interpolate nans
    best_learning_curve = best_learning_curve.interpolate(method='linear')
    return best_learning_curve.loc[flop_intervals]

mlp_df = mlp_df[mlp_df['scaling'] == 'width']
mlp_df["base_N"] = mlp_df["base_N"].fillna(-1)
mlp_df["g"] = mlp_df["target_N"] / mlp_df["base_N"]
mlp_df["N"] = mlp_df["target_params_m"] * 10 ** 6
mlp_df["D"] = mlp_df["cfg_target_tokens_per_param"] * mlp_df["N"]
mlp_df["Validation Loss"] = mlp_df["curve_val"].map(lambda x: x[-1])

mlp_df["C"] = 6 * mlp_df["D"] * mlp_df["N"]
# errors = ((mlp_df["C"] - mlp_df["max_flops"]) / mlp_df["max_flops"]).map(lambda x: abs(x) > 0.01)
# mlp_df[errors]

mlp_df = mlp_df.loc[mlp_df.groupby(by=["N", "D", "base_N", "method"])["Validation Loss"].idxmin()]
mlp_df["tokens"] = mlp_df["D"]

upper_limit = sorted(mlp_df["target_N"].unique())[-3]
lower_limit = sorted(mlp_df["target_N"].unique())[1]
mlp_df = mlp_df[(mlp_df["target_N"] >= lower_limit) & (mlp_df["target_N"] <= upper_limit)]


In [ ]:
mlp_df["target_scale"].unique()

In [ ]:
n2n_mlp_df = mlp_df[(mlp_df['method']=='net2net')]
n2n_mlp_jump_df = filter_pairs(n2n_mlp_df, jump_size=1)
n2n_mlp_jump_df["target_N"] = n2n_mlp_jump_df["N"]

n2n_mlp_data_X, n2n_mlp_data_y = preprocess_approach3_data(n2n_mlp_jump_df, log_transform_y=True)
n2n_mlp_data_X.to_csv("./output/n2n_mlp_data_X.csv")
n2n_mlp_data_y.to_csv("./output/n2n_mlp_data_y.csv")
n2n_mlp_params = fit_chin3(n2n_mlp_data_X, n2n_mlp_data_y)

y_pred = functional_form_chin3(n2n_mlp_data_X[:,:2], n2n_mlp_params)
n2n_mlp_r2 = r2_score(np.exp(n2n_mlp_data_y), np.exp(y_pred))

In [ ]:
mup_mlp_df = mlp_df[(mlp_df['method']=='scratch')]
mup_mlp_df["target_N"] = mup_mlp_df["N"]

mup_mlp_data_X, mup_mlp_data_y = preprocess_approach3_data(mup_mlp_df, log_transform_y=True)
mup_mlp_data_X.to_csv("./output/mup_mlp_data_X.csv")
mup_mlp_data_y.to_csv("./output/mup_mlp_data_y.csv")
mup_mlp_params = fit_chin3(mup_mlp_data_X, mup_mlp_data_y)

y_pred = functional_form_chin3(mup_mlp_data_X[:,:2], mup_mlp_params)
mup_mlp_r2 = r2_score(np.exp(mup_mlp_data_y), np.exp(y_pred))

In [ ]:
snp_mlp_df = mlp_df[(mlp_df['method']=='snp')]
snp_mlp_jump_df = filter_pairs(snp_mlp_df, jump_size=1)
snp_mlp_jump_df["target_N"] = snp_mlp_jump_df["N"]

snp_mlp_data_X, snp_mlp_data_y = preprocess_approach3_data(snp_mlp_jump_df, log_transform_y=True)
snp_mlp_data_X.to_csv("./output/snp_mlp_data_X.csv")
snp_mlp_data_y.to_csv("./output/snp_mlp_data_y.csv")
snp_mlp_params = fit_chin3(snp_mlp_data_X, snp_mlp_data_y)

y_pred = functional_form_chin3(snp_mlp_data_X[:,:2], snp_mlp_params)
n2n_mlp_r2 = r2_score(np.exp(snp_mlp_data_y), np.exp(y_pred))

In [ ]:
grid_steps = 250

D_contour = np.linspace(snp_mlp_data_X[:,1].min()*0.85, snp_mlp_data_X[:,1].max()*10, grid_steps)
N_contour = np.linspace(snp_mlp_data_X[:,0].min()*0.85, snp_mlp_data_X[:,0].max()*10, grid_steps)

N_cg_snp_mlp , D_cg_snp_mlp = np.meshgrid(N_contour, D_contour)
L_cg_snp_mlp = np.zeros_like(N_cg_snp_mlp)

for params, factor in [(mup_mlp_params, -1), (snp_mlp_params, 1)]:
    _L_cg_snp_mlp = np.exp(functional_form_chin3(
            np.column_stack((
                N_cg_snp_mlp.reshape(-1),
                D_cg_snp_mlp.reshape(-1)
            )),
            params
        ))
    L_cg_snp_mlp += _L_cg_snp_mlp.reshape(L_cg_snp_mlp.shape) * factor


D_contour = np.linspace(n2n_mlp_data_X[:,1].min()*0.85, n2n_mlp_data_X[:,1].max()*10, grid_steps)
N_contour = np.linspace(n2n_mlp_data_X[:,0].min()*0.85, n2n_mlp_data_X[:,0].max()*10, grid_steps)

N_cg_n2n_mlp , D_cg_n2n_mlp = np.meshgrid(N_contour, D_contour)
L_cg_n2n_mlp = np.zeros_like(N_cg_n2n_mlp)

for params, factor in [(mup_mlp_params, -1), (n2n_mlp_params, 1)]:
    _L_cg_n2n_mlp = np.exp(functional_form_chin3(
            np.column_stack((
                N_cg_n2n_mlp.reshape(-1),
                D_cg_n2n_mlp.reshape(-1)
            )),
            params
        ))
    L_cg_n2n_mlp += _L_cg_n2n_mlp.reshape(L_cg_n2n_mlp.shape) * factor

In [ ]:
import seaborn as sns
sns.set_style("white")
sns.set_context("notebook", font_scale=1.475)
fig, axes = plt.subplots(1, 2, figsize=(5*2, 4.75), layout='constrained')

vmin = float(min(L_cg_n2n_mlp.min(), L_cg_snp_mlp.min()))
vmax = float(max(L_cg_n2n_mlp.max(), L_cg_snp_mlp.max()))
norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
levels = np.linspace(vmin, vmax, 100)
cmap = "RdBu_r"

# SZP
ax = axes[0]
cs = ax.contourf(N_cg_snp_mlp, D_cg_snp_mlp, L_cg_snp_mlp, levels=levels, cmap=cmap, norm=norm, zorder=-1)
ax.plot(N_contour, N_contour*20, color='black', linewidth=2, zorder=-1, alpha=0.3)
lime_N_D = ax.contour(N_cg_snp_mlp, D_cg_snp_mlp, L_cg_snp_mlp, levels=[0], colors='lime', linewidths=4, zorder=1, alpha=0.5)
# ax.contour(N_cg, D_cg, L_cg, levels=[0], colors='green', linewidths=1.5, zorder=1)

# ax.scatter(snp_mlp_data_X[:,0], snp_mlp_data_X[:,1], color="black")
L_diff_df = get_loss_diff(snp_mlp_data_X, snp_mlp_data_y, mup_mlp_data_X, mup_mlp_data_y)
ax.scatter(L_diff_df["N"], L_diff_df["D"], c=L_diff_df["L_diff"], cmap=cmap, norm=norm, edgecolor="k", linewidth=0.5,)
ax.loglog()
ax.set_title("SZP")

# N2N
ax = axes[1]
cs = ax.contourf(N_cg_n2n_mlp, D_cg_n2n_mlp, L_cg_n2n_mlp, levels=levels, cmap=cmap, norm=norm, zorder=-1)
ax.plot(N_contour, N_contour*20, color='black', linewidth=2, zorder=-1, alpha=0.3)
lime_N_D = ax.contour(N_cg_n2n_mlp, D_cg_n2n_mlp, L_cg_n2n_mlp, levels=[0], colors='lime', linewidths=4, zorder=1, alpha=0.5)
# ax.contour(N_cg, D_cg, L_cg, levels=[0], colors='green', linewidths=1.5, zorder=1)

# ax.scatter(n2n_mlp_data_X[:,0], n2n_mlp_data_X[:,1], color="black")
L_diff_df = get_loss_diff(n2n_mlp_data_X, n2n_mlp_data_y, mup_mlp_data_X, mup_mlp_data_y)
ax.scatter(L_diff_df["N"], L_diff_df["D"], c=L_diff_df["L_diff"], cmap=cmap, norm=norm, edgecolor="k", linewidth=0.5,)
ax.loglog()
ax.set_title("Net2Net")


fig.colorbar(cs, ax=ax, label=r'Loss Difference ($\text{warmstart} - \mu P$)')
fig.supxlabel("Parameters")
fig.supylabel("Tokens")
fig.suptitle("MLP")
fig.savefig(f"figures/neurips_chinchilla_mlp_low_resolution.pdf")

In [ ]:
grid_steps = 250

D_contour = np.linspace(warm_data_X[:,1].min()*0.85, warm_data_X[:,1].max()*10, grid_steps)
N_contour = np.linspace(warm_data_X[:,0].min()*0.85, warm_data_X[:,0].max()*10, grid_steps)

N_cg_llm , D_cg_llm = np.meshgrid(N_contour, D_contour)
L_cg_llm = np.zeros_like(N_cg_llm)

for params, factor in [(n2n_params, -1), (warm_params, 1)]:
    _L_cg_llm = np.exp(functional_form_chin3(
            np.column_stack((
                N_cg_llm.reshape(-1),
                D_cg_llm.reshape(-1)
            )),
            params
        ))
    L_cg_llm += _L_cg_llm.reshape(N_cg_warm.shape) * factor


D_contour = np.linspace(n2n_mlp_data_X[:,1].min()*0.85, n2n_mlp_data_X[:,1].max()*10, grid_steps)
N_contour = np.linspace(n2n_mlp_data_X[:,0].min()*0.85, n2n_mlp_data_X[:,0].max()*10, grid_steps)

N_cg_mlp , D_cg_mlp = np.meshgrid(N_contour, D_contour)
L_cg_mlp = np.zeros_like(N_cg_mlp)

for params, factor in [(n2n_mlp_params, -1), (snp_mlp_params, 1)]:
    _L_cg_mlp = np.exp(functional_form_chin3(
            np.column_stack((
                N_cg_mlp.reshape(-1),
                D_cg_mlp.reshape(-1)
            )),
            params
        ))
    L_cg_mlp += _L_cg_mlp.reshape(L_cg_n2n_mlp.shape) * factor

In [ ]:
import seaborn as sns
sns.set_style("white")
sns.set_context("notebook", font_scale=1.475)
fig, axes = plt.subplots(1, 2, figsize=(5.5*2, 4.75), layout='constrained')


levels = 100
cmap = "RdBu_r"

# SZP
ax = axes[1]
vmin = L_cg_llm.min()
vmax = L_cg_llm.max()
norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

cs = ax.contourf(N_cg_llm, D_cg_llm, L_cg_llm, levels=levels, cmap=cmap, norm=norm, zorder=-1)
ax.plot(np.linspace(warm_data_X[:,0].min()*0.85, warm_data_X[:,0].max()*10, grid_steps), np.linspace(warm_data_X[:,0].min()*0.85, warm_data_X[:,0].max()*10, grid_steps)*20, color='black', linewidth=2, zorder=-1, alpha=0.3)
lime_N_D = ax.contour(N_cg_llm, D_cg_llm, L_cg_llm, levels=[0], colors='lime', linewidths=4, zorder=1, alpha=0.5)
# ax.contour(N_cg, D_cg, L_cg, levels=[0], colors='green', linewidths=1.5, zorder=1)

# ax.scatter(snp_mlp_data_X[:,0], snp_mlp_data_X[:,1], color="black")
L_diff_df = get_loss_diff(warm_data_X, warm_data_y, n2n_data_X, n2n_data_y)
ax.scatter(L_diff_df["N"], L_diff_df["D"], c=L_diff_df["L_diff"], cmap=cmap, norm=norm, edgecolor="k", linewidth=0.5,)
ax.loglog()
ax.set_title("LLM")
fig.colorbar(cs, ax=ax, label=r'Loss Difference ($\text{SZP} -  \text{Net2Net}$)')
# N2N
ax = axes[0]
vmin = L_cg_mlp.min()
vmax = L_cg_mlp.max()
norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

cs = ax.contourf(N_cg_mlp, D_cg_mlp, L_cg_mlp, levels=levels, cmap=cmap, norm=norm, zorder=-1)
ax.plot(N_contour, N_contour*20, color='black', linewidth=2, zorder=-1, alpha=0.3)
lime_N_D = ax.contour(N_cg_mlp, D_cg_mlp, L_cg_mlp, levels=[0], colors='lime', linewidths=4, zorder=1, alpha=0.5)
# ax.contour(N_cg, D_cg, L_cg, levels=[0], colors='green', linewidths=1.5, zorder=1)

# ax.scatter(n2n_mlp_data_X[:,0], n2n_mlp_data_X[:,1], color="black")
L_diff_df = get_loss_diff(snp_mlp_data_X, snp_mlp_data_y, n2n_mlp_data_X, n2n_mlp_data_y)
ax.scatter(L_diff_df["N"], L_diff_df["D"], c=L_diff_df["L_diff"], cmap=cmap, norm=norm, edgecolor="k", linewidth=0.5,)
ax.loglog()
ax.set_title("MLP")

fig.colorbar(cs, ax=ax, label=r'Loss Difference ($\text{SZP} -  \text{Net2Net}$)')

fig.supxlabel("Parameters")
fig.supylabel("Tokens")
fig.savefig(f"figures/neurips_chinchilla_mlp_low_resolution.pdf")

# Summary Table

In [ ]:
data = [
    format_approach3_results(warm_params),
    format_approach3_results(n2n_params),
    format_approach3_results(mup_params),
    format_approach3_results(snp_mlp_params),
    format_approach3_results(n2n_mlp_params),
    format_approach3_results(mup_mlp_params),
]
columns = [r"$A$", r"$\alpha$", r"$B$",  r"$\beta$", r"$E$"]
table_df = pd.DataFrame(data, columns=columns).T
table_df.loc[r"$R^2$"] = [f"{warm_r2:0.4f}", f"{n2n_r2:0.4f}", f"{mup_r2:0.4f}", f"{snp_mlpr_r2:0.4f}", f"{n2n_r2:0.4f}", f"{mup_r2:0.4f}"]
table_df.columns = [r"\ours{}", "Net2Net", "Scratch", r"\ours{}", "Net2Net", "Scratch"]
latex = table_df.to_latex()
latex = latex.replace(r'$R^2$', r'\midrule' + '\n' + r'$R^2$')
print(latex)

In [ ]:
warm_data_X

In [ ]:
mup_data_y

In [ ]:
mup_df = final_points_df[final_points_df['method']=='mup']
mup_jump_df = filter_pairs(mup_df, jump_size=2)
mup_data_X, mup_data_y = preprocess_approach3_data(mup_jump_df)
mup_params = fit_chin3(mup_data_X, mup_data_y)

In [ ]:
warm_df = final_points_df[(final_points_df['method']=='paws') & (final_points_df['shrinking']==0.4)]
warm_jump_df = filter_pairs(warm_df, jump_size=2)
warm_data_X, warm_data_y = preprocess_approach3_data(warm_jump_df)
warm_params = fit_chin3(warm_data_X, warm_data_y)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4.5), layout='constrained')

D_contour = np.linspace(final_points_df["tokens"].min()*0.85, 2e12, 300)
N_contour = np.linspace(final_points_df["target_N"].min()*0.85, 1e10, 300)

N_cg , D_cg = np.meshgrid(N_contour, D_contour)
L_cg = np.zeros_like(N_cg)



for params, factor in [(mup_params, -1), (warm_params, 1)]:
    _L_cg = np.exp(functional_form_chin3(
            np.column_stack((
                N_cg.reshape(-1),
                D_cg.reshape(-1)
            )),
            params
        ))
    L_cg += _L_cg.reshape(N_cg.shape) * factor


cs = ax.contourf(N_cg, D_cg, L_cg, levels=30, cmap="RdBu_r", norm=norm, zorder=-1)
ax.plot(N_contour, N_contour*20, color='black', linewidth=2, zorder=-1, alpha=0.3)
lime_N_D = ax.contour(N_cg, D_cg, L_cg, levels=[0], colors='lime', linewidths=4, zorder=1, alpha=0.5)
# ax.contour(N_cg, D_cg, L_cg, levels=[0], colors='green', linewidths=1.5, zorder=1)

ax.scatter(warm_data_X[:,0], warm_data_X[:,1], color="black")

fig.colorbar(cs, ax=ax, label=r'Loss Difference ($\text{warmstart} - \mu P$)')
ax.set_xlabel("Parameters")
ax.set_ylabel("Tokens")
ax.loglog()

In [ ]:
mup_df = final_points_df[final_points_df['method']=='mup']
mup_jump_df = filter_pairs(mup_df, jump_size=4)
mup_data_X, mup_data_y = preprocess_approach3_data(mup_jump_df)
mup_params = fit_chin3(mup_data_X, mup_data_y)

In [ ]:
warm_df = final_points_df[(final_points_df['method']=='paws') & (final_points_df['shrinking']==0.4)]
warm_jump_df = filter_pairs(warm_df, jump_size=4)
warm_data_X, warm_data_y = preprocess_approach3_data(warm_jump_df)
warm_params = fit_chin3(warm_data_X, warm_data_y)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4.5), layout='constrained')

D_contour = np.linspace(final_points_df["tokens"].min()*0.85, 2e12, 300)
N_contour = np.linspace(final_points_df["target_N"].min()*0.85, 1e10, 300)

N_cg , D_cg = np.meshgrid(N_contour, D_contour)
L_cg = np.zeros_like(N_cg)



for params, factor in [(mup_params, -1), (warm_params, 1)]:
    _L_cg = np.exp(functional_form_chin3(
            np.column_stack((
                N_cg.reshape(-1),
                D_cg.reshape(-1)
            )),
            params
        ))
    L_cg += _L_cg.reshape(N_cg.shape) * factor


cs = ax.contourf(N_cg, D_cg, L_cg, levels=30, cmap="RdBu_r", norm=norm, zorder=-1)
ax.plot(N_contour, N_contour*20, color='black', linewidth=2, zorder=-1, alpha=0.3)
lime_N_D = ax.contour(N_cg, D_cg, L_cg, levels=[0], colors='lime', linewidths=4, zorder=1, alpha=0.5)
# ax.contour(N_cg, D_cg, L_cg, levels=[0], colors='green', linewidths=1.5, zorder=1)

ax.scatter(warm_data_X[:,0], warm_data_X[:,1], color="black")

fig.colorbar(cs, ax=ax, label=r'Loss Difference ($\text{warmstart} - \mu P$)')
ax.set_xlabel("Parameters")
ax.set_ylabel("Tokens")
ax.loglog()

In [ ]:
# Create visualizations for custom loss function
# Try fitting and look at R2 and extrapolation

In [ ]:
# Misfitting
LBFGS with different tolerrances

a_vals = np.arange(0, 25, 1)
b_vals = np.arange(0, 25, 1)
e_vals = np.arange(-1, 1, 0.1)
alpha_vals = np.arange(0, 2, 0.1)
beta_vals = np.arange(0, 2, 0.1)

a_vals = np.arange(0, 25, 5)
b_vals = np.arange(0, 25, 5)
e_vals = np.arange(-1, 1, 0.5)
alpha_vals = np.arange(0, 2, 0.5)
beta_vals = np.arange(0, 2, 0.5)


# Gemstone
method="L-BFGS-B"

for _ in range(num_parameters):
        param_search_array.append(np.arange(0, 2.5, 0.5))  # exp, 5 possible values
    for _ in range(num_parameters):
        param_search_array.append(np.arange(0, 30, 5))  # coefficient, 6 values
    param_search_array.append(np.arange(-1, 1.5, 0.5))  # error, 5 values



# mine

np.linspace(0, 15, 5),  # a
np.linspace(0., 1., 5),  # alpha
np.linspace(0, 15, 5),  # b
np.linspace(0., 1., 5),  # beta
np.linspace(0., 1., 5),  # e


# Farseer

torch.optim.LBFGS([params],
lr=1e-1,
history_size=10,
max_iter=20,
line_search_fn="strong_wolfe")

for a in np.linspace(0, 25, 5):
        for b in np.linspace(0, 25, 5):
            for e in np.linspace(-1, 1, 4):
                for alpha in np.linspace(0, 2, 4):